---
## Por que Parquet e nao CSV?

Antes de coletar dados, precisamos saber **onde salva-los e em qual formato**.  
A resposta e Parquet — e aqui esta o motivo.

| | CSV | Parquet |
|---|---|---|
| Tipos de dados | Perde ao salvar | Preserva |
| Tamanho | Grande | Comprimido |
| Leitura parcial | Le tudo | Le so o que precisa |

---
#### - Instalando os pacotes necessários para trabalharmos com os dados:

In [ ]:
# instação da lib pandas

# manipulação de tabelas (DataFrames)
!pip install pandas~=2.1
# backend para leitura/escrita de arquivos Parquet
!pip install pyarrow~=17.0
# sistema de arquivos unificado (local, S3, GCS)
!pip install fsspec~=2023.10

## Por que instalar pyarrow?

O pandas sabe trabalhar com DataFrames, mas para ler e gravar Parquet ele precisa de um motor por trás.

Um dos motores mais usados é o `pyarrow`.Sem ele, você pode receber um erro parecido com:

```codex- t
ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparqu
```

Em uma frase:

> `pyarrow` é o motor que permite ao pandas trabalhar bem com arquivos Parquet, que são formatos mais eficientes para pipelines de dados. 

```

## Por que instalar o fsspec?

O `fsspec` significa algo próximo de **filesystem specification**.

> `fsspec` é uma camada que ajuda bibliotecas de dados a acessar arquivos em diferentes sistemas, como disco local, nuvem, buckets e outros armazenamentos, usando uma interface parecida.

Ele funciona como uma camada de abstração para acessar arquivos em diferentes lugares usando uma interface parecida.

Normalmente, quando usamos pandas, lemos arquivos locais assim:

```python
import pandas as pd

df = pd.read_csv("data/clientes.csv")
```

Mas em projetos maiores, os arquivos podem estar em outros lugares, como:

```text
s3://meu-bucket/clientes.csv
gcs://meu-bucket/clientes.parquet
az://container/dados.parquet
```

O `fsspec` ajuda bibliotecas como pandas, DVC e outras ferramentas a lidar com esses caminhos de forma mais flexível.

Ele permite que o código trate diferentes tipos de armazenamento de maneira mais padronizada.

---

## Exemplo 1 - Tipos de Dados

### CSV

In [1]:
import pandas as pd

# Criando um DataFrame simples para a demonstracao
df = pd.DataFrame({
    'customer_id': ['C001', 'C002'],
    'churned':     [1, 0],
    'signup_date': pd.to_datetime(['2026-01-01', '2026-02-15'])
})

print('=== DataFrame original ===')
print(df.dtypes)
print()

# PROBLEMA: CSV perde os tipos de dados
df.to_csv('data/teste.csv', index=False)
df_csv = pd.read_csv('data/teste.csv')
print('=== Apos salvar e ler como CSV ===')
print(df_csv.dtypes)
print('\n"signup_date" virou:', df_csv['signup_date'].dtype)  # object = string!
print()

=== DataFrame original ===
customer_id            object
churned                 int64
signup_date    datetime64[ns]
dtype: object

=== Apos salvar e ler como CSV ===
customer_id    object
churned         int64
signup_date    object
dtype: object

"signup_date" virou: object



In [ ]:
# Cálculo com data com o dataframe original
print('Dias com assinatura dataframe original: ', (pd.Timestamp.today() - df['signup_date']).dt.days)

# Cálculo com data com o csv
print('Dias com assinatura CSV: ', (pd.Timestamp.today() - df_csv['signup_date']).dt.days)

### Parquet

In [ ]:
# SOLUCAO: Parquet preserva os tipos
df.to_parquet('data/teste.parquet', index=False)
df_pq = pd.read_parquet('data/teste.parquet')
print('=== Apos salvar e ler como Parquet ===')
print(df_pq.dtypes)
print('signup_date e:', df_pq['signup_date'].dtype)  # datetime64 — correto!

In [ ]:
# Cálculo com data com o dataframe original
print('Dias com assinatura dataframe original: ', (pd.Timestamp.today() - df['signup_date']).dt.days)

# Cálculo com data com o csv
print('Dias com assinatura parquet: ', (pd.Timestamp.today() - df_pq['signup_date']).dt.days)

---
### Exemplo 2 - Tamanho

In [2]:
import os
import time

# Criando um DataFrame um pouco maior para o exemplo fazer sentido
linhas = 1_000_000

df = pd.DataFrame({
    "customer_id": [f"C{i:07d}" for i in range(linhas)],
    "churned": [1, 0] * (linhas // 2),
    "signup_date": pd.date_range("2026-03-01", periods=linhas, freq="min"),
    "monthly_spend": [99.90, 49.90, 79.90, 29.90] * (linhas // 4),
    "plan": ["Basic", "Premium", "Standard", "Basic"] * (linhas // 4),
})

# Adicionando várias colunas extras para simular um arquivo mais largo
for i in range(30):
    df[f"feature_{i}"] = i

df.head()

,customer_id,churned,signup_date,monthly_spend,plan,feature_0,feature_1,feature_2,feature_3,feature_4,...,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29
0,C0000000,1,2026-03-01 00:00:00,99.9,Basic,0,1,2,3,4,...,20,21,22,23,24,25,26,27,28,29
1,C0000001,0,2026-03-01 00:01:00,49.9,Premium,0,1,2,3,4,...,20,21,22,23,24,25,26,27,28,29
2,C0000002,1,2026-03-01 00:02:00,79.9,Standard,0,1,2,3,4,...,20,21,22,23,24,25,26,27,28,29
3,C0000003,0,2026-03-01 00:03:00,29.9,Basic,0,1,2,3,4,...,20,21,22,23,24,25,26,27,28,29
4,C0000004,1,2026-03-01 00:04:00,99.9,Basic,0,1,2,3,4,...,20,21,22,23,24,25,26,27,28,29


In [3]:
# Verificando informações do Dataframe
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 35 columns):
 #   Column         Non-Null Count    Dtype         
---  ------         --------------    -----         
 0   customer_id    1000000 non-null  object        
 1   churned        1000000 non-null  int64         
 2   signup_date    1000000 non-null  datetime64[ns]
 3   monthly_spend  1000000 non-null  float64       
 4   plan           1000000 non-null  object        
 5   feature_0      1000000 non-null  int64         
 6   feature_1      1000000 non-null  int64         
 7   feature_2      1000000 non-null  int64         
 8   feature_3      1000000 non-null  int64         
 9   feature_4      1000000 non-null  int64         
 10  feature_5      1000000 non-null  int64         
 11  feature_6      1000000 non-null  int64         
 12  feature_7      1000000 non-null  int64         
 13  feature_8      1000000 non-null  int64         
 14  feature_9      1000000 non-null  in

In [4]:
# Salvando em CSV
df.to_csv("data/clientes.csv", index=False)
# Salvando em Parquet
df.to_parquet("data/clientes.parquet", index=False)

# Comparando tamanho dos arquivos
csv_size = os.path.getsize("data/clientes.csv") / (1024 * 1024)
parquet_size = os.path.getsize("data/clientes.parquet") / (1024 * 1024)

print(f"Tamanho CSV: {csv_size:.2f} MB")
print(f"Tamanho Parquet: {parquet_size:.2f} MB")

Tamanho CSV: 118.49 MB
Tamanho Parquet: 11.41 MB


---
### Exemplo 3 - Leitura Parcial

## CSV

In [5]:
inicio = time.time()

df_csv_colunas = pd.read_csv(
    "data/clientes.csv",
    usecols=["customer_id", "monthly_spend"]
)

fim = time.time()

t_csv = fim - inicio

print(df_csv_colunas.head())
print(f"Tempo lendo colunas do CSV: {t_csv:.4f} segundos")

  customer_id  monthly_spend
0    C0000000           99.9
1    C0000001           49.9
2    C0000002           79.9
3    C0000003           29.9
4    C0000004           99.9
Tempo lendo colunas do CSV: 1.8644 segundos


## Parquet

In [6]:
# Lendo apenas algumas colunas do Parquet
inicio = time.time()

df_parquet_colunas = pd.read_parquet(
    "data/clientes.parquet",
    columns=["customer_id", "monthly_spend"]
)

fim = time.time()

t_parquet = fim - inicio

print(df_parquet_colunas.head())
print(f"Tempo lendo apenas colunas do Parquet: {t_parquet:.4f} segundos")

  customer_id  monthly_spend
0    C0000000           99.9
1    C0000001           49.9
2    C0000002           79.9
3    C0000003           29.9
4    C0000004           99.9
Tempo lendo apenas colunas do Parquet: 0.6435 segundos


In [7]:
# Diferença de tempo entre ambos os arquivos
print(f'No exemplo o arquivo parquet foi: ~{round(t_parquet/t_csv, 2) * 100}% mais rápido que o CSV')

No exemplo o arquivo parquet foi: ~35.0% mais rápido que o CSV


In [7]:
def decoracao(func):
    def acao_a_realizar():
        print('babilonia')
        resultado = func()
        print('babilônia time')
        return resultado
    return acao_a_realizar


@decoracao
def decorar():
    print('babilônia em chamas')

In [8]:
decorar()

babilonia
babilônia em chamas
babilônia time
